# STCL Locking Workflow with RedPitayaSTCL


This notebook provides a complete desktop workflow for setting up and operating the **Scanning Transfer Cavity Lock (STCL)** using the `RedPitayaSTCL` codebase.

It is organized as:
1. Clone + install + basic software validation.
2. Verify Red Pitaya OS / SSH / SCPI services.
3. Generate a **triangular ramp on OUT2** and acquire on **IN1**.
4. Run STCL in sequence:
   - Connect and start STCL server.
   - Start cavity scan.
   - Start cavity lock.
   - Start laser lock.

> Update all IP addresses and lock parameters before running on hardware.

## 3) Verify signal path: generate triangular ramp (OUT2) and acquire on IN1

This test confirms end-to-end analog functionality:
- Configure SCPI generator on **OUT2** with triangle waveform.
- Trigger acquisition and read **IN1** samples.
- Plot the captured signal.

### Wiring
- Connect `OUT2` to `IN1` on the same Red Pitaya (directly or via attenuator as needed).
- Ensure input gain/jumper settings match expected voltage range.

## 4) Connect and start STCL server

The next cells use `LockClient` / `RP_client` from this repository.

### Expected Red Pitaya roles
- `Cav`: cavity scan/master (`mode="scan"`)
- `Lock1`: laser lock RP (`mode="lock"`)
- `Mon`: monitor RP (`mode="monitor"`)

You can add more lock RPs by extending the dictionary.

> If this step appears to run forever after "connected to all redpitayas", one RP likely failed to start `RunLock.py` cleanly. The code cell below adds a timeout and troubleshooting hints.


# Scanning the cavity

In [1]:
import sys
sys.path.append("./RedPitayaSTCL")

In [2]:
from pathlib import Path
Path()

WindowsPath('.')

In [3]:
import time
import threading
from lockclient import LockClient, RP_client

In [4]:
# --- Hardware address configuration ---
RP_CAV_IP = "192.168.0.101"   # scanning cavity RP
# RP_LOCK1_IP = "192.168.0.102" # first laser lock RP
# RP_MON_IP = "192.168.0.100"   # monitor RP

RPs = [RP_CAV_IP]

RP_SSH_USER = "root"
RP_SSH_PASS = "root"
SCPI_PORT = 5000

In [5]:
RPs = {
    "Cav": RP_client((RP_CAV_IP, 5000), {}, mode="scan"),
    # "Lock1": RP_client((RP_LOCK1_IP, 5000), {}, mode="lock"),
    # "Mon": RP_client((RP_MON_IP, 5000), {}, mode="monitor"),
}

Lock = LockClient(RPs)

In [6]:
def _wrap(fn, err):
    try:
        fn()
    except Exception as exc:
        err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start()
    t.join(timeout=timeout_s)

    if t.is_alive():
        raise TimeoutError(
            f"{name} timed out after {timeout_s}s.\n"
            "Possible fixes:\n"
            "- Confirm each RP is reachable over SSH and credentials are correct.\n"
            "- On each RP run: python3 /home/jupyter/RedPitaya/RunLock.py and check for missing python packages/errors.\n"
            "- Ensure port 5000 is free and not blocked by firewall/VPN.\n"
            "- Power-cycle the stuck RP and retry connect_all()."
        )

    if "exc" in err:
        raise RuntimeError(f"{name} failed: {err['exc']}")

In [7]:
# --- connect to Red Pitayas ---
run_with_timeout(Lock.connect_all, timeout_s=30, name="Lock.connect_all")
print("Connected to all Red Pitayas")

connecting...
Connected to all Red Pitayas


In [8]:
# --- start STCL event loop in background ---
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    print("Event loop started")
else:
    print("Event loop already running")

# --- allow modules to initialize ---
time.sleep(2)

Event loop started


In [9]:
# --- check connection status ---
status = {name: rp.connected for name, rp in Lock.RPs.items()}
print("Connection states:", status)

Connection states: {'Cav': True}


In [10]:
# --- inspect available modules (debugging) ---
print("Available modules:")
Lock.retrieve_settings("Cav")

Available modules:


{'Master': {'PID': {'P': 0.1, 'I': 50, 'D': 0.0},
  'range': [[781, 3124], [13281, 15624]],
  'lockpoint': 1.8,
  'enabled': True,
  'dec': 16,
  'peak_finder': {'name': 'SG_deriv', 'window_size': 21, 'order': 1}}}

In [11]:
Lock.start_scan("Cav")

Inside start_scan!

Inside start_loop!

Thread defined!

Daemon True!

Thread started!




Exception occured during connection: [WinError 10057] A request to send or receive data was disallowed because the socket is not connected and (when sending on a datagram socket using a sendto call) no address was supplied


### Giving errors

In [ ]:
# ::::::::::::::::::::::::::::::::::::::::::::::   Errors in this block. Debug befor running! ::::::::::::::::::::::::::::::::::


# --- update cavity parameters safely ---
if "Cav" in Lock.RPs and "Master" in Lock.retrieve_settings('Cav'):

    Lock.update_setting("Cav", "Master", "range", [[0.10, 0.40], [1.70, 2.00]])
    Lock.update_setting("Cav", "Master", "lockpoint", 1.80)
    Lock.update_setting("Cav", "Master", "enabled", True)

    Lock.update_setting(
        "Cav",
        "Master",
        "PID",
        {"P": 0.08, "I": 0.001, "D": 0.0, "limit": [-1.0, 1.0]},
    )

    print("Cavity parameters sent")

else:
    print("Cavity module not available in STCL configuration")


print("Cavity parameters updated!")

In [14]:
Lock.connect('Cav')
Lock.start_scan("Cav")

Cav already connected.


# Later

## 5) Start cavity scan

Use this section to:
- set cavity/reference ranges and lockpoint,
- run cavity scanning loop,
- start/stop monitor RP.

In [ ]:
# Parameter updates for cavity peak finding / stabilization
Lock.update_setting("Cav", "Master", "range", [[0.10, 0.40], [1.70, 2.00]])
Lock.update_setting("Cav", "Master", "lockpoint", 1.80)
Lock.update_setting("Cav", "Master", "enabled", True)

# Optional PID tuning for cavity lock loop
Lock.update_setting("Cav", "Master", "PID", {"P": 0.08, "I": 0.001, "D": 0.0, "limit": [-1.0, 1.0]})

print("Cavity parameters sent")

In [ ]:
# Start/stop cavity scan and monitor
Lock.start_scan("Cav")
Lock.start_monitor("Mon")   # opens live cavity monitor window

# Manual actions when needed:
# Lock.stop_monitor("Mon")
# Lock.stop_loop("Cav")

## 6) Start cavity lock

When scan alignment is good, stop free scan loop and start cavity lock.

In [ ]:
# Stop scan loop first, then lock cavity
Lock.stop_loop("Cav")
Lock.start_lock("Cav")
print("Cavity lock command issued")

# To stop cavity lock:
# Lock.stop_loop("Cav")

## 7) Start laser lock

Configure lock channels (`Slave1` / `Slave2`) and enable locking outputs.

In [ ]:
# Example: configure and enable Lock1 -> Slave1 (OUT1)
Lock.update_setting("Lock1", "Slave1", "enabled", True)
Lock.update_setting("Lock1", "Slave1", "range", [0.50, 0.80])
Lock.update_setting("Lock1", "Slave1", "lockpoint", 0.60)
Lock.update_setting("Lock1", "Slave1", "PID", {"P": 0.05, "I": 0.0008, "D": 0.0, "limit": [-1.0, 1.0]})

# Optional second laser channel on same RP
Lock.update_setting("Lock1", "Slave2", "enabled", False)
# Lock.update_setting("Lock1", "Slave2", "range", [1.05, 1.25])
# Lock.update_setting("Lock1", "Slave2", "lockpoint", 1.15)
# Lock.update_setting("Lock1", "Slave2", "PID", {"P": 0.04, "I": 0.0005, "D": 0.0, "limit": [-1.0, 1.0]})

# Start / stop laser lock loop on RP
Lock.start_lock("Lock1")
print("Laser lock started")

# To stop laser lock:
# Lock.stop_loop("Lock1")

## 8) Monitoring and safe shutdown

Use error monitoring for lightweight long-duration tracking, then shut everything down cleanly.

In [ ]:
# Optional error monitor
# Lock.stop_monitor("Mon")
# Lock.start_error_monitor("Mon", tmin=30e-3)

# Stop individual loops manually if needed:
# Lock.stop_loop("Lock1")
# Lock.stop_loop("Cav")

# Full cleanup (recommended at end of session)
# Lock.close()

---
### Notes
- If any RP becomes unresponsive, power-cycle it and rerun the connection cells.
- Keep cavity scan and lock ranges conservative first, then tighten after stable operation.
- Save tuned JSON settings from `settings/*.json` for reproducible startup.